<div style="direction: rtl; white-space: normal; line-height: 1;">
بارگذاری مدل و کتابخانه 
</div>

In [1]:
# Load model, dataset, and saved embeddings

from pathlib import Path
import json
import numpy as np
from sentence_transformers import SentenceTransformer

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

dataset_file = project_root / "data" / "processed" / "quran_dataset_clean.json"
embeddings_file = project_root / "data" / "embeddings" / "quran_embeddings.npy"

with open(dataset_file, "r", encoding="utf-8") as f:
    quran_data = json.load(f)

quran_embeddings = np.load(embeddings_file)

model_name = "intfloat/multilingual-e5-small"
model = SentenceTransformer(model_name, device="cpu")

print("Records:", len(quran_data))
print("Embeddings shape:", quran_embeddings.shape)
print("Model:", model_name)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Records: 6236
Embeddings shape: (6236, 384)
Model: intfloat/multilingual-e5-small


<div style="direction: rtl; white-space: normal; line-height: 1;">
ساخت تابع Retrieval
</div>

In [2]:
# Semantic retrieval function

def retrieve_verses(query, top_k=5):
    """
    Retrieve the most relevant Quran verses for a user query
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    top_k = min(top_k, len(quran_data))

    query_embedding = model.encode(
        [f"query: {query.strip()}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]

    scores = quran_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = quran_data[index]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "surah": item["surah"],
            "ayah": item["ayah"],
            "arabic": item["arabic"],
            "fooladvand": item["fooladvand"],
            "ansarian": item["ansarian"],
        })

    return results


print("Retrieval function is ready.")

Retrieval function is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، تابع بازیابی را با یک پرسش فارسی آزمایش می‌کنیم و پنج آیه مرتبط‌تر را همراه با امتیاز شباهت نمایش می‌دهیم.
</div>

In [3]:
# Test the retrieval function

query = "خداوند توبه‌کنندگان را دوست دارد"

results = retrieve_verses(query, top_k=5)

print("Query:", query)
print()

for result in results:
    print(f"Rank {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Surah: {result['surah']} | Ayah: {result['ayah']}")
    print("Arabic:", result["arabic"])
    print("Fooladvand:", result["fooladvand"])
    print("Ansarian:", result["ansarian"])
    print("-" * 80)

Query: خداوند توبه‌کنندگان را دوست دارد

Rank 1
Score: 0.8901
Surah: 40 | Ayah: 7
Arabic: ٱلَّذِينَ يَحْمِلُونَ ٱلْعَرْشَ وَمَنْ حَوْلَهُۥ يُسَبِّحُونَ بِحَمْدِ رَبِّهِمْ وَيُؤْمِنُونَ بِهِۦ وَيَسْتَغْفِرُونَ لِلَّذِينَ ءَامَنُوا۟ رَبَّنَا وَسِعْتَ كُلَّ شَىْءٍ رَّحْمَةً وَعِلْمًا فَٱغْفِرْ لِلَّذِينَ تَابُوا۟ وَٱتَّبَعُوا۟ سَبِيلَكَ وَقِهِمْ عَذَابَ ٱلْجَحِيمِ
Fooladvand: کسانی که عرش [خدا] را حمل می‌کنند، و آنها که پیرامون آنند، به سپاس پروردگارشان تسبیح می‌گویند و به او ایمان دارند و برای کسانی که گرویده‌اند طلب آمرزش می‌کنند: «پروردگارا، رحمت و دانش [تو بر] هر چیز احاطه دارد؛ کسانی را که توبه کرده و راه تو را دنبال کرده‌اند ببخش و آنها را از عذاب آتش نگاه دار.»
Ansarian: فرشتگانی که عرش را حمل می کنند و آنان که پیرامون آن هستند، همراه سپاس و ستایش، پروردگارشان را تسبیح می گویند و به او ایمان دارند و برای اهل ایمان آمرزش می طلبند، [و می گویند:] پروردگارا! از روی رحمت و دانش همه چیز را فرا گرفته ای، پس آنان را که توبه کرده اند و راه تو را پیروی نموده اند بیامرز، و آنان را از عذاب دوز

<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، بازیابی را با یک پرسش دارای آیه مشخص آزمایش می‌کنیم تا بررسی شود آیا آیه بقره ۲۲۲ در نتایج برتر قرار می‌گیرد یا نه.
</div>

In [4]:
# Test retrieval with a known Quran verse

query = "خدا توبه کنندگان را دوست دارد"

results = retrieve_verses(query, top_k=10)

print("Query:", query)
print()

for result in results:
    marker = " <-- Expected verse" if (
        result["surah"] == 2 and result["ayah"] == 222
    ) else ""

    print(
        f"Rank {result['rank']} | "
        f"Score: {result['score']:.4f} | "
        f"Surah: {result['surah']} | "
        f"Ayah: {result['ayah']}{marker}"
    )
    print("Fooladvand:", result["fooladvand"])
    print("-" * 80)

Query: خدا توبه کنندگان را دوست دارد

Rank 1 | Score: 0.8804 | Surah: 40 | Ayah: 7
Fooladvand: کسانی که عرش [خدا] را حمل می‌کنند، و آنها که پیرامون آنند، به سپاس پروردگارشان تسبیح می‌گویند و به او ایمان دارند و برای کسانی که گرویده‌اند طلب آمرزش می‌کنند: «پروردگارا، رحمت و دانش [تو بر] هر چیز احاطه دارد؛ کسانی را که توبه کرده و راه تو را دنبال کرده‌اند ببخش و آنها را از عذاب آتش نگاه دار.»
--------------------------------------------------------------------------------
Rank 2 | Score: 0.8770 | Surah: 61 | Ayah: 4
Fooladvand: در حقیقت، خدا دوست دارد کسانی را که در راه او صف در صف، چنانکه گویی بنایی ریخته شده از سرب‌اند، جهاد می‌کنند.
--------------------------------------------------------------------------------
Rank 3 | Score: 0.8761 | Surah: 2 | Ayah: 222 <-- Expected verse
Fooladvand: از تو در باره عادت ماهانه [زنان‌] می‌پرسند، بگو: «آن، رنجی است. پس هنگام عادت ماهانه، از [آمیزش با] زنان کناره گیری کنید، و به آنان نزدیک نشوید تا پاک شوند. پس چون پاک شدند، از همان جا که خدا به شما فر

<div style="direction: rtl; white-space: normal; line-height: 1;">
این تست نشان داد Retrieval موضوع «توبه» را می‌فهمد، اما عبارت دقیق «دوست داشتن توبه‌کنندگان» را خوب بالا نمی‌آورد. احتمالاً چون متن embedding هر آیه شامل عربی + دو ترجمه + اطلاعات سوره و آیه است و مفهوم دقیق فارسی کمی رقیق شده.

فعلاً embedding را دوباره نمی‌سازیم. اول رتبه واقعی آیه بقره ۲۲۲ را پیدا می‌کنیم.



در این بخش، امتیاز و رتبه دقیق آیه بقره ۲۲۲ را در میان تمام آیات بررسی می‌کنیم تا میزان فاصله آن از نتایج برتر مشخص شود.
</div>

In [5]:
# Inspect the exact rank of Surah 2, Ayah 222

query = "خدا توبه کنندگان را دوست دارد"

query_embedding = model.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)[0]

scores = quran_embeddings @ query_embedding
sorted_indices = np.argsort(scores)[::-1]

target_index = next(
    index
    for index, item in enumerate(quran_data)
    if item["surah"] == 2 and item["ayah"] == 222
)

target_rank = int(np.where(sorted_indices == target_index)[0][0]) + 1
target_item = quran_data[target_index]

print("Target rank:", target_rank)
print("Target score:", round(float(scores[target_index]), 4))
print("Arabic:", target_item["arabic"])
print("Fooladvand:", target_item["fooladvand"])
print("Ansarian:", target_item["ansarian"])

Target rank: 3
Target score: 0.8761
Arabic: وَيَسْـَٔلُونَكَ عَنِ ٱلْمَحِيضِ قُلْ هُوَ أَذًى فَٱعْتَزِلُوا۟ ٱلنِّسَآءَ فِى ٱلْمَحِيضِ وَلَا تَقْرَبُوهُنَّ حَتَّىٰ يَطْهُرْنَ فَإِذَا تَطَهَّرْنَ فَأْتُوهُنَّ مِنْ حَيْثُ أَمَرَكُمُ ٱللَّهُ إِنَّ ٱللَّهَ يُحِبُّ ٱلتَّوَّٰبِينَ وَيُحِبُّ ٱلْمُتَطَهِّرِينَ
Fooladvand: از تو در باره عادت ماهانه [زنان‌] می‌پرسند، بگو: «آن، رنجی است. پس هنگام عادت ماهانه، از [آمیزش با] زنان کناره گیری کنید، و به آنان نزدیک نشوید تا پاک شوند. پس چون پاک شدند، از همان جا که خدا به شما فرمان داده است، با آنان آمیزش کنید.» خداوند توبه‌کاران و پاکیزگان را دوست می‌دارد.
Ansarian: از تو درباره حیض می پرسند، بگو: حیض، حالت ناملایم و زیان باری است؛ پس در زمان حیض از [آمیزش با] زنان کناره گیری کنید، و با آنان نزدیکی ننمایید تا پاک شوند؛ و هنگامی که پاک شدند از جایی که خدا به شما فرمان داده با آنان آمیزش کنید. یقیناً خدا کسانی را که بسیار توبه می کنند، و کسانی را که خود را [با پذیرش انواع پاکی ها از همه آلودگی ها] پاکیزه می کنند. دوست دارد.


<div style="direction: rtl; white-space: normal; line-height: 1;">
این نتیجه قابل قبول نیست؛ رتبه ۲۴۸ یعنی ساختار فعلی embedding باید اصلاح شود.

علت اصلی این است که داخل هر passage این‌ها را با هم گذاشتیم:

متن بلند عربی
دو ترجمه
شماره سوره و آیه

در نتیجه مفهوم دقیق ترجمه فارسی رقیق شده است.

برای سؤال‌های فارسی، فعلاً embedding را فقط از دو ترجمه فارسی می‌سازیم و عربی را برای نمایش نتیجه نگه می‌داریم.

توضیح قبل از کد:

در این بخش، متن‌های مخصوص بازیابی فارسی را فقط از دو ترجمه قرآن آماده می‌کنیم تا مفاهیم فارسی دقیق‌تر بازیابی شوند.

برمیگردیم به نوت بوک چهار و امبدینگ فارسی را تغییر میدهیم
</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">
تغییر امبدینگ فارسی موثر بود و نتیجه از رتبه ۲۴۸ به ۳ رسید

و برای سوال اول هم ایه ۲۲۲ رتبه ۲ شده 
پس بازیابی فعلی قابل قبول است وهر چند هنوز میشود با جستجوی ترکیبی کیورد و سمنتیک بهترش کرد.
</div>